# TranAD 在 WADI 上的复现实验

本笔记本与仓库根目录下 `TranAD/` 中的 `main.py`、`preprocess.py`、`src/models.py` 等逻辑一致，用于在 **WADI** 数据上训练与评估 **TranAD** 模型。

**环境与依赖（`TranAD/requirements.txt`）**

1. **先确认当前 Jupyter 用的 Python 里能 `import torch`**。若没有 PyTorch，在终端（建议先 `conda activate` 你跑笔记本的环境）执行例如：
   ```text
   pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
   ```
   （需要 GPU 版请到 [pytorch.org](https://pytorch.org) 选对应命令。）

2. **再安装 TranAD 其余库**（与仓库一致）：
   ```text
   cd TranAD
   pip install -r requirements.txt
   ```
   其中包含：`pandas`、`tqdm`、`matplotlib`、`dgl`、`numpy`、`SciencePlots`、`scikit-learn`、`xlrd` 等；`requirements.txt` 内也有简短注释。

3. **内核**：选「已装好上述包」的那个 Python / conda 环境作为 Notebook kernel 即可。

**数据**

- 将官方 WADI 放到 `TranAD/data/WADI/`，或在下文配置 **自定义 CSV**（`Date`/`Time` + 第 4 列起为传感器；你也可用「测试集带点级标签列」的 A2 流程）。
- 预处理生成 `TranAD/processed/WADI/{train,test,labels}.npy`；训练时通过 `os.chdir(TranAD)` 读写 `processed/` 与 `checkpoints/`。

环境与运行说明见项目根目录 README.md。

## 1. 配置：TranAD 根目录与命令行参数

`--retrain` 表示重新训练；若已有 `checkpoints/TranAD_WADI/model.ckpt` 且只想测试，可改为 `--test`（并去掉 `--retrain`）。

In [ ]:
from pathlib import Path
import os
import os
import sys
from pathlib import Path

# 修改为本地 TranAD 仓库绝对路径
TRANAD_ROOT = Path(os.environ.get("TRANAD_ROOT", "../TranAD")).resolve()

assert TRANAD_ROOT.is_dir(), f"未找到 TranAD 目录: {TRANAD_ROOT}"

os.chdir(TRANAD_ROOT)
if str(TRANAD_ROOT) not in sys.path:
    sys.path.insert(0, str(TRANAD_ROOT))

# 须在首次 import TranAD 内模块之前设置（与命令行等价）
sys.argv = [
    "tranad_wadi_reproduce.ipynb",
    "--dataset", "WADI",
    "--model", "TranAD",
    "--retrain",
]

print("cwd:", os.getcwd())
print("argv:", sys.argv)

## 2.（可选）自定义 WADI 文件路径并写入 `processed/WADI/`

**情况 A — 官方三文件**（`WADI_14days.csv`、`WADI_attackdata.csv`、`WADI_attacklabels.csv`）：在下方把 `CUSTOM_*` 设为路径，或把文件放进 `TranAD/data/WADI/` 后运行 `python preprocess.py WADI`。

**情况 B — 仅训练 CSV + 带标签的测试 CSV**（例如 iTrust A2：`WADI_14days_new.csv` 与 `WADI_attackdataLABLE.csv`，最后一列为点级标签，**1=正常、-1=攻击**）：设置 `WADI_A2_DIR` 并令 `USE_WADI_POINT_LABEL_CSV = True`。脚本会跳过首行无效表头、对齐 `Row/Date/Time` 后第 4 列起的传感器，并按 TranAD 同样规则 **每 10 行取 1 行** 与 **按列 min-max 归一化**（与 `convertNumpy` 一致）。

In [ ]:
from pathlib import Path
import os
from pathlib import Path
from typing import List, Optional, Tuple, Union

import numpy as np
import pandas as pd

from src.folderconstants import output_folder


def convertNumpy(df: pd.DataFrame) -> np.ndarray:
    x = df[df.columns[3:]].values[::10, :]
    return (x - x.min(0)) / (x.ptp(0) + 1e-4)


def preprocess_wadi_custom(
    train_csv: Path,
    test_csv: Path,
    attack_labels_csv: Path,
    train_skiprows: int = 1000,
    train_nrows: Union[int, float] = 2e5,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """与 preprocess.py 中 WADI 分支一致，输出 train/test/labels 数组。"""
    ls = pd.read_csv(attack_labels_csv)
    train = pd.read_csv(train_csv, skiprows=train_skiprows, nrows=int(train_nrows))
    test = pd.read_csv(test_csv)
    train.dropna(how="all", inplace=True)
    test.dropna(how="all", inplace=True)
    train.fillna(0, inplace=True)
    test.fillna(0, inplace=True)
    test["Time"] = test["Time"].astype(str)
    test["Time"] = pd.to_datetime(test["Date"] + " " + test["Time"])
    labels = test.copy(deep=True)
    for i in test.columns.tolist()[3:]:
        labels[i] = 0
    for i in ["Start Time", "End Time"]:
        ls[i] = ls[i].astype(str)
        ls[i] = pd.to_datetime(ls["Date"] + " " + ls[i])
    for index, row in ls.iterrows():
        to_match = row["Affected"].split(", ")
        matched = []
        for i in test.columns.tolist()[3:]:
            for tm in to_match:
                if tm in i:
                    matched.append(i)
                    break
        st, et = str(row["Start Time"]), str(row["End Time"])
        labels.loc[(labels["Time"] >= st) & (labels["Time"] <= et), matched] = 1
    train_a, test_a, lab_a = convertNumpy(train), convertNumpy(test), convertNumpy(labels)
    print("train, test, labels:", train_a.shape, test_a.shape, lab_a.shape)
    return train_a, test_a, lab_a


def _norm_downsample_10(arr: np.ndarray) -> np.ndarray:
    x = arr[::10, :].astype(np.float64)
    return (x - x.min(0)) / (x.ptp(0) + 1e-4)


def preprocess_wadi_point_label_csv(
    train_csv: Path,
    test_csv: Path,
    train_nrows: Optional[Union[int, float]] = 2e5,
    train_skiprows: int = 0,
    test_skiprows: Optional[List[int]] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """测试 CSV 含点级标签列（1=正常，-1=攻击）；自动跳过测试文件首行无效表头。"""
    if test_skiprows is None:
        test_skiprows = [0]
    nrows = int(train_nrows) if train_nrows is not None else None
    train = pd.read_csv(train_csv, skiprows=train_skiprows, nrows=nrows)
    test = pd.read_csv(test_csv, skiprows=test_skiprows)
    train.columns = train.columns.str.strip()
    test.columns = test.columns.str.strip()
    label_col = None
    for c in test.columns:
        low = str(c).lower()
        if "attack" in low and ("lable" in low or "label" in low):
            label_col = c
            break
    if label_col is None:
        label_col = test.columns[-1]
    feats = list(train.columns[3:])
    miss = [f for f in feats if f not in test.columns]
    if miss:
        raise ValueError("测试集缺少与训练集对齐的传感器列（示例）: " + repr(miss[:5]))
    train_x = train[feats].fillna(0).values.astype(np.float64)
    test_x = test[feats].fillna(0).values.astype(np.float64)
    train_a = _norm_downsample_10(train_x)
    test_a = _norm_downsample_10(test_x)
    raw_lab = test[label_col].values[::10]
    raw_lab = pd.to_numeric(pd.Series(raw_lab), errors="coerce").fillna(1.0).values
    y = (raw_lab < 0).astype(np.float64)
    labels = np.tile(y.reshape(-1, 1), (1, test_a.shape[1]))
    print("train, test, labels:", train_a.shape, test_a.shape, labels.shape, "| label列:", label_col)
    return train_a, test_a, labels


# ---------- A2：训练 CSV + 带标签测试 CSV（你的路径）----------
WADI_A2_DIR = Path(os.environ.get("WADI_DATA_ROOT", "../data/WADI"))
USE_WADI_POINT_LABEL_CSV = True
TRAIN_NROWS = 2e5
TRAIN_SKIPROWS = 0

# ---------- 官方三文件（与上互斥：改用官方时设 USE_WADI_POINT_LABEL_CSV = False）----------
CUSTOM_TRAIN = None
CUSTOM_TEST = None
CUSTOM_LABELS = None

folder = Path(output_folder) / "WADI"
folder.mkdir(parents=True, exist_ok=True)

if USE_WADI_POINT_LABEL_CSV:
    tr_path = WADI_A2_DIR / "WADI_14days_new.csv"
    te_path = WADI_A2_DIR / "WADI_attackdataLABLE.csv"
    if not tr_path.is_file() or not te_path.is_file():
        raise FileNotFoundError(f"请检查路径: {tr_path} , {te_path}")
    tr, te, lb = preprocess_wadi_point_label_csv(
        tr_path, te_path, train_nrows=TRAIN_NROWS, train_skiprows=TRAIN_SKIPROWS
    )
    np.save(folder / "train.npy", tr)
    np.save(folder / "test.npy", te)
    np.save(folder / "labels.npy", lb)
    print("已写入:", folder)
elif CUSTOM_TRAIN is not None and CUSTOM_TEST is not None and CUSTOM_LABELS is not None:
    tr, te, lb = preprocess_wadi_custom(CUSTOM_TRAIN, CUSTOM_TEST, CUSTOM_LABELS)
    np.save(folder / "train.npy", tr)
    np.save(folder / "test.npy", te)
    np.save(folder / "labels.npy", lb)
    print("已写入:", folder)
else:
    print("未启用 USE_WADI_POINT_LABEL_CSV 且未设置三文件 CUSTOM_*。请运行 preprocess.py 或打开 USE_WADI_POINT_LABEL_CSV。")

## 3. 若尚未生成 `processed/WADI`，在终端执行预处理

在 `TranAD` 目录下：
```bash
python preprocess.py WADI
```

下面单元格会检查 `processed/WADI` 是否存在。

In [ ]:
from pathlib import Path
import os
from pathlib import Path

wadi_proc = Path("processed") / "WADI"
need = [wadi_proc / "train.npy", wadi_proc / "test.npy", wadi_proc / "labels.npy"]
missing = [p for p in need if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "缺少预处理文件: " + ", ".join(str(p) for p in missing)
        + "\n请在 TranAD 目录运行: python preprocess.py WADI\n"
        "或完成上一节的自定义路径预处理。"
    )
print("预处理文件就绪:", need)

## 4. 导入模块（依赖上文的 `sys.argv`）

若修改了 `--dataset` / `--model`，请**重启内核**后从第一节重新运行，避免 `constants` 缓存旧参数。

**顺序**：先完成 **4.1 自检** 一格，再运行 **import main** 那一格。

### 4.1 环境自检（DGL → torchdata.datapipes）\n
\n
先运行 **下面代码格** 做自检；通过后再运行 **再下一格** `import main`。若终端已装 torchdata 仍报错，请确认内核为 **(tsad)**。

In [ ]:
# Install dependencies explicitly using docs/reproduction.md before running.
import subprocess
import torchdata.datapipes
import pydantic


### 4.15 PyTorch 2.x 兼容补丁（is_causal）

若运行第 5 节训练时报错 `forward() got an unexpected keyword argument 'is_causal'`，说明 PyTorch 2.x 与 TranAD 不兼容。**必须先运行下面代码格**，再执行第 5 节。

In [ ]:
# TranAD 使用 src.dlutils 中的自定义 Encoder/Decoder Layer，须先导入再打补丁
import src.dlutils as dlutils

# PyTorch 2.x 的 nn.TransformerEncoder/Decoder 会传入 is_causal，dlutils 中的 Layer 签名不支持
_enc_orig = dlutils.TransformerEncoderLayer.forward
def _enc_patched(self, src, src_mask=None, src_key_padding_mask=None, **kwargs):
    kwargs.pop("is_causal", None)
    return _enc_orig(self, src, src_mask, src_key_padding_mask)
dlutils.TransformerEncoderLayer.forward = _enc_patched

_dec_orig = dlutils.TransformerDecoderLayer.forward
def _dec_patched(self, tgt, memory, tgt_mask=None, memory_mask=None,
                 tgt_key_padding_mask=None, memory_key_padding_mask=None, **kwargs):
    kwargs.pop("is_causal", None)
    return _dec_orig(self, tgt, memory, tgt_mask, memory_mask,
                     tgt_key_padding_mask, memory_key_padding_mask)
dlutils.TransformerDecoderLayer.forward = _dec_patched

print("PyTorch is_causal 兼容补丁已应用（dlutils TransformerEncoder/DecoderLayer），可继续运行第 5 节。")

In [ ]:
import pickle
import importlib

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from time import time
from pprint import pprint
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, TensorDataset

# 确保 constants 使用当前 argv（首次运行通常无需 reload）
import src.constants as constants
importlib.reload(constants)

from src.constants import *
from src.pot import pot_eval
from src.diagnosis import hit_att, ndcg
from src.plotting import plotter
from src.utils import plot_accuracies

from main import (
    load_dataset,
    convert_to_windows,
    backprop,
    save_model,
    load_model,
)

constants.preds.clear()
print("dataset:", args.dataset, "model:", args.model, "retrain:", args.retrain, "test:", args.test)

## 5. 训练与测试（与 `main.py` 主流程一致）

- 训练 5 个 epoch（与仓库默认相同）。
- 评估使用 POT + `src/constants.py` 中 WADI 的 `lm` 与 `lr` 等超参。

In [ ]:
train_loader, test_loader, labels = load_dataset(args.dataset)

model, optimizer, scheduler, epoch, accuracy_list = load_model(args.model, labels.shape[1])

trainD, testD = next(iter(train_loader)), next(iter(test_loader))
trainO, testO = trainD, testD
trainD, testD = convert_to_windows(trainD, model), convert_to_windows(testD, model)

if not args.test:
    print(f"Training {args.model} on {args.dataset}")
    num_epochs = 5
    start = time()
    for e in tqdm(list(range(epoch + 1, epoch + num_epochs + 1)), desc="epochs"):
        lossT, lr = backprop(e, model, trainD, trainO, optimizer, scheduler)
        accuracy_list.append((lossT, lr))
    print("Training time: {:10.4f} s".format(time() - start))
    save_model(model, optimizer, scheduler, e, accuracy_list)
    plot_accuracies(accuracy_list, f"{args.model}_{args.dataset}")

model.eval()
print(f"Testing {args.model} on {args.dataset}")
with torch.no_grad():
    loss, y_pred = backprop(0, model, testD, testO, optimizer, scheduler, training=False)

if not args.test:
    if "TranAD" in model.name:
        testO_plot = torch.roll(testO, 1, 0)
    else:
        testO_plot = testO
    plotter(f"{args.model}_{args.dataset}", testO_plot, y_pred, loss, labels)

df = pd.DataFrame()
with torch.no_grad():
    lossT, _ = backprop(0, model, trainD, trainO, optimizer, scheduler, training=False)
for i in range(loss.shape[1]):
    lt, l, ls = lossT[:, i], loss[:, i], labels[:, i]
    result, pred = pot_eval(lt, l, ls)
    constants.preds.append(pred)
    df = pd.concat([df, pd.DataFrame([result])], ignore_index=True)

lossTfinal, lossFinal = np.mean(lossT, axis=1), np.mean(loss, axis=1)
labelsFinal = (np.sum(labels, axis=1) >= 1) + 0
result, _ = pot_eval(lossTfinal, lossFinal, labelsFinal)
result.update(hit_att(loss, labels))
result.update(ndcg(loss, labels))
print(df)
pprint(result)

## 6. 内核说明与结果可视化

- **可以继续使用同一个 Jupyter kernel**（同一套 Python 解释器与环境），不必为画图单独换内核。
- 若你**修改了**第 1 节里的 `sys.argv`（例如 `--dataset` / `--model`），为避免 `constants` 仍用旧参数，建议 **Restart Kernel** 后从第 1 节顺序重跑。
- 下面画图依赖 **上一节「训练与测试」已成功执行**，内存中已有 `accuracy_list`、`lossFinal`、`labelsFinal`、`result` 等。若刚重启过内核，请先重跑预处理与训练节。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ---------- 1) 训练损失（每个 epoch 的 L1）----------
try:
    al = accuracy_list
    if al is not None and len(al) > 0:
        fig, ax = plt.subplots(figsize=(8, 3.2))
        ax.plot([x[0] for x in al], marker="o", lw=1)
        ax.set_xlabel("epoch")
        ax.set_ylabel("train L1")
        ax.set_title("TranAD training loss")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
except NameError:
    print("未找到 accuracy_list：请先运行「训练与测试」单元格。")

# ---------- 2) 测试集：各维平均后的异常分数 + 真实异常区间 ----------
try:
    scores = np.asarray(lossFinal, dtype=np.float64).ravel()
    y = np.asarray(labelsFinal, dtype=np.float64).ravel()
    t = np.arange(len(scores))
    fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    axes[0].plot(t, scores, lw=0.4, color="C0")
    axes[0].set_ylabel("mean score")
    axes[0].set_title("Test: aggregate anomaly score (mean over dims)")
    axes[0].grid(True, alpha=0.3)
    axes[1].fill_between(t, 0, 1, where=(y > 0.5), color="red", alpha=0.25, label="ground-truth anomaly")
    axes[1].set_ylim(-0.1, 1.1)
    axes[1].set_ylabel("label")
    axes[1].set_xlabel("time index (after downsampling)")
    axes[1].legend(loc="upper right")
    plt.tight_layout()
    plt.show()
except NameError as e:
    print("缺少 lossFinal / labelsFinal：请先运行训练单元格。", e)

# ---------- 3) 分数 + POT 阈值线 ----------
try:
    thr = float(result["threshold"])
    scores = np.asarray(lossFinal, dtype=np.float64).ravel()
    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.plot(scores, lw=0.35, color="C0", label="mean score")
    ax.axhline(thr, color="C1", ls="--", lw=1.2, label=f"POT threshold = {thr:.6f}")
    ax.set_xlabel("time index")
    ax.set_ylabel("score")
    ax.set_title("Aggregate score vs POT threshold")
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except (NameError, KeyError) as e:
    print("无法绘制阈值图：请先完成训练且 result 中含 threshold。", e)

# ---------- 4) 正常 vs 异常 分数分布（便于观察可分性）----------
try:
    s = np.asarray(lossFinal, dtype=np.float64).ravel()
    y = np.asarray(labelsFinal, dtype=np.float64).ravel()
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.hist(s[y < 0.5], bins=60, alpha=0.55, density=True, label="normal")
    ax.hist(s[y > 0.5], bins=60, alpha=0.55, density=True, label="anomaly")
    ax.set_xlabel("score")
    ax.set_ylabel("density")
    ax.set_title("Score distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()
except NameError:
    print("跳过分布图：缺少 lossFinal / labelsFinal。")

### 6.1 混淆矩阵与扩展可视化

下方代码格会输出：**混淆矩阵热力图**、**POT 评估指标表格**、**ROC 曲线**、**PR 曲线**、**异常分数时序片段对比**、**TPR/FPR 随阈值变化** 等图表。需先运行第 5 节以生成 `lossFinal`、`labelsFinal`、`result`。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["font.sans-serif"] = ["SimHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import numpy as np
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score
)

try:
    from src.pot import pot_eval
except ImportError:
    pot_eval = None

try:
    scores = np.asarray(lossFinal, dtype=np.float64).ravel()
    y_true = np.asarray(labelsFinal, dtype=np.float64).ravel()
    _, y_pred = pot_eval(lossTfinal, lossFinal, labelsFinal)

    fig_all = []

    # ----- 1) 混淆矩阵（数量 + 归一化） -----
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
    fig1, axes = plt.subplots(1, 2, figsize=(10, 4))
    im0 = axes[0].imshow(cm, cmap="Blues")
    axes[0].set_xticks([0, 1])
    axes[0].set_yticks([0, 1])
    axes[0].set_xticklabels(["正常", "异常"])
    axes[0].set_yticklabels(["正常", "异常"])
    axes[0].set_xlabel("预测")
    axes[0].set_ylabel("真实")
    axes[0].set_title("混淆矩阵（数量）")
    for i in range(2):
        for j in range(2):
            axes[0].text(j, i, str(cm[i, j]), ha="center", va="center")
    im1 = axes[1].imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    axes[1].set_xticks([0, 1])
    axes[1].set_yticks([0, 1])
    axes[1].set_xticklabels(["正常", "异常"])
    axes[1].set_yticklabels(["正常", "异常"])
    axes[1].set_xlabel("预测")
    axes[1].set_ylabel("真实")
    axes[1].set_title("混淆矩阵（行归一化）")
    for i in range(2):
        for j in range(2):
            axes[1].text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center")
    plt.tight_layout()
    plt.show()
    fig_all.append("混淆矩阵")

    # ----- 2) POT 指标表格 -----
    print("\n--- POT 评估指标 ---")
    for k, v in result.items():
        if isinstance(v, (int, float)) and not isinstance(v, bool):
            print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
    print(f"  TP={result.get('TP', '')} TN={result.get('TN', '')} FP={result.get('FP', '')} FN={result.get('FN', '')}\n")

    # ----- 3) ROC 曲线 -----
    fpr, tpr, _ = roc_curve(y_true, scores)
    roc_auc_val = auc(fpr, tpr)
    fig2, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, lw=2, label=f"ROC (AUC = {roc_auc_val:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("假阳性率 (FPR)")
    ax.set_ylabel("真阳性率 (TPR)")
    ax.set_title("ROC 曲线")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    fig_all.append("ROC")

    # ----- 4) PR 曲线 -----
    prec, rec, _ = precision_recall_curve(y_true, scores)
    ap = average_precision_score(y_true, scores)
    fig3, ax = plt.subplots(figsize=(5, 4))
    ax.plot(rec, prec, lw=2, label=f"PR (AP = {ap:.3f})")
    ax.set_xlabel("召回率 (Recall)")
    ax.set_ylabel("精确率 (Precision)")
    ax.set_title("Precision-Recall 曲线")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    fig_all.append("PR")

    # ----- 5) 异常分数 vs 真实标签（时序片段，前 2000 点） -----
    n_show = min(2000, len(scores))
    fig4, ax = plt.subplots(figsize=(12, 3))
    t = np.arange(n_show)
    ax.fill_between(t, 0, 1, where=y_true[:n_show] > 0.5, alpha=0.25, color="red", label="真实异常")
    ax.plot(t, scores[:n_show], lw=0.5, color="C0", label="异常分数")
    ax.set_xlabel("时间步")
    ax.set_ylabel("异常分数")
    ax.set_title("异常分数 vs 真实标签（前 {} 步）".format(n_show))
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    fig_all.append("时序对比")

    # ----- 6) 预测 vs 真实（时序片段） -----
    fig5, axes = plt.subplots(2, 1, figsize=(12, 3.5), sharex=True)
    axes[0].plot(t, y_true[:n_show], drawstyle="steps-post", color="C1", label="真实标签")
    axes[0].set_ylabel("真实")
    axes[0].legend(loc="upper right")
    axes[0].set_ylim(-0.1, 1.2)
    axes[1].plot(t, y_pred[:n_show], drawstyle="steps-post", color="C2", label="预测标签")
    axes[1].set_ylabel("预测")
    axes[1].set_xlabel("时间步")
    axes[1].legend(loc="upper right")
    axes[1].set_ylim(-0.1, 1.2)
    axes[0].set_title("真实 vs 预测（POT 阈值，前 {} 步）".format(n_show))
    plt.tight_layout()
    plt.show()
    fig_all.append("预测对比")

    # ----- 7) 不同阈值下的 F1 / Precision / Recall -----
    thresholds = np.percentile(scores, np.linspace(1, 99, 50))
    f1s, precs, recs = [], [], []
    for th in thresholds:
        pred_t = (scores > th).astype(float)
        tp = np.sum((pred_t == 1) & (y_true > 0.5))
        fp = np.sum((pred_t == 1) & (y_true < 0.5))
        fn = np.sum((pred_t == 0) & (y_true > 0.5))
        p = tp / (tp + fp + 1e-8)
        r = tp / (tp + fn + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        f1s.append(f1)
        precs.append(p)
        recs.append(r)
    fig6, ax = plt.subplots(figsize=(7, 4))
    ax.plot(thresholds, f1s, "o-", ms=3, label="F1")
    ax.plot(thresholds, precs, "s-", ms=3, label="Precision")
    ax.plot(thresholds, recs, "^-", ms=3, label="Recall")
    ax.axvline(result.get("threshold", np.median(thresholds)), color="gray", ls="--", label="POT 阈值")
    ax.set_xlabel("阈值")
    ax.set_ylabel("分数")
    ax.set_title("F1 / Precision / Recall 随阈值变化")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    fig_all.append("阈值曲线")

    print("已绘制:", ", ".join(fig_all))

except NameError as e:
    print("缺少 lossFinal / labelsFinal / lossTfinal / result：请先运行第 5 节「训练与测试」。", e)

## 7. 等价的命令行复现

```bash
cd TranAD
python preprocess.py WADI
python main.py --model TranAD --dataset WADI --retrain
```

后续可在本笔记本同目录其他实验中继续做对比学习等扩展。